# 03 - Data Cleaning

**Purpose:** Clean and validate the extracted dataset for analysis.

| | |
|---|---|
| **Inputs** | `data/pilot/pilot_500.parquet` |
| **Outputs** | `data/processed/pilot_clean.parquet`, `figures/missing_data.png`, `figures/distributions.png` |
| **Runtime** | ~10 seconds |

## Steps
1. Load extracted data
2. Missing data analysis
3. Range checks and validation
4. Distribution inspection
5. Severity score calculation
6. Export cleaned dataset

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from severity_scale import add_severity_column, DEFAULT_WEIGHTS

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
df = pd.read_parquet('../data/pilot/pilot_500.parquet')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Missing Data Analysis

In [ ]:
# Null counts per field
key_fields = [
    'gender', 'age', 'education', 'occupation', 'employed',
    'family_size', 'num_children', 'prior_criminal',
    'sentence_type', 'sentence_months', 'sentence_fine_mnt',
    'court', 'judge', 'crime_article', 'case_date',
]

missing = pd.DataFrame({
    'null_count': df[key_fields].isnull().sum(),
    'null_pct': (df[key_fields].isnull().sum() / len(df) * 100).round(1),
    'non_null': df[key_fields].notna().sum(),
})
missing = missing.sort_values('null_pct', ascending=False)
print(missing.to_string())

In [ ]:
# Visualize missingness
fig, ax = plt.subplots(figsize=(10, 6))
missing_pct = missing['null_pct'].sort_values(ascending=True)
missing_pct.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% Missing')
ax.set_title('Missing Data by Field (Pilot 500 Cases)')
for i, v in enumerate(missing_pct.values):
    ax.text(v + 0.5, i, f'{v:.0f}%', va='center')
plt.tight_layout()
plt.savefig('../figures/missing_data.png', dpi=150)
plt.show()

In [ ]:
# Cases with no demographics at all (bio section not found)
no_demo = df[df['gender'].isna() & df['age'].isna() & df['education'].isna()]
print(f"Cases with no demographics: {len(no_demo)} ({len(no_demo)/len(df)*100:.1f}%)")

# Cases with no sentence info
no_sent = df[df['sentence_type'].isna()]
print(f"Cases with no sentence type: {len(no_sent)} ({len(no_sent)/len(df)*100:.1f}%)")

# Usable cases (have both gender and sentence_type)
usable = df[df['gender'].notna() & df['sentence_type'].notna()]
print(f"Usable cases (gender + sentence_type): {len(usable)} ({len(usable)/len(df)*100:.1f}%)")

## 3. Range Checks and Validation

In [ ]:
# Age range check
if df['age'].notna().any():
    print("=== Age ===")
    print(df['age'].describe())
    out_of_range = df[(df['age'] < 14) | (df['age'] > 90)]
    if len(out_of_range) > 0:
        print(f"\nOut of range (14-90): {len(out_of_range)} cases")
        print(out_of_range[['case_id', 'age']].to_string())
    else:
        print("All ages within 14-90 range")

In [ ]:
# Category validation
print("=== Gender values ===")
print(df['gender'].value_counts(dropna=False))
print()

print("=== Education values ===")
print(df['education'].value_counts(dropna=False))
print()

print("=== Sentence type values ===")
print(df['sentence_type'].value_counts(dropna=False))

In [ ]:
# Cross-variable logic checks
# Fine amount should only exist for fine sentences
fines_without_type = df[
    (df['sentence_fine_mnt'].notna()) & 
    (df['sentence_type'] != 'fine')
]
print(f"Fine amounts without 'fine' sentence type: {len(fines_without_type)}")

# Imprisonment months should only exist for imprisonment/suspended
months_check = df[
    (df['sentence_months'].notna()) & 
    (~df['sentence_type'].isin(['imprisonment', 'suspended']))
]
print(f"Months without imprisonment/suspended type: {len(months_check)}")

# Fine amount sanity check (should be reasonable MNT values)
if df['sentence_fine_mnt'].notna().any():
    print(f"\nFine amount range: {df['sentence_fine_mnt'].min():,.0f} - {df['sentence_fine_mnt'].max():,.0f} MNT")
    print(f"Fine amount median: {df['sentence_fine_mnt'].median():,.0f} MNT")

## 4. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Age distribution
ax = axes[0, 0]
df['age'].dropna().hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.set_title('Defendant Age Distribution')

# Gender distribution
ax = axes[0, 1]
df['gender'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_ylabel('Count')
ax.set_title('Gender Distribution')
ax.tick_params(axis='x', rotation=0)

# Sentence type distribution
ax = axes[1, 0]
df['sentence_type'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Count')
ax.set_title('Sentence Type Distribution')
ax.tick_params(axis='x', rotation=45)

# Education distribution
ax = axes[1, 1]
edu_order = ['бага', 'суурь', 'бүрэн дунд', 'тусгай дунд', 'дээд']
edu_counts = df['education'].value_counts().reindex(edu_order).dropna()
edu_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Count')
ax.set_title('Education Level Distribution')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/distributions.png', dpi=150)
plt.show()

## 5. Severity Score Calculation

Convert sentence types to a unified "imprisonment-month equivalent" scale:
- Imprisonment: 1.0x months
- Suspended: 0.5x months  
- Probation: 0.3x months
- Community service: hours / 160
- Fine: handled separately (conversion TBD)

In [ ]:
# Rename community service column for severity function
df_sev = df.copy()
if 'sentence_community_hours' in df_sev.columns:
    df_sev['community_service_hours'] = df_sev['sentence_community_hours']

df_sev = add_severity_column(df_sev)

print("Severity score by sentence type:")
print(df_sev.groupby('sentence_type')['severity'].describe().round(1))
print()
print(f"Total with severity: {df_sev['severity'].notna().sum()}")
print(f"Total without severity: {df_sev['severity'].isna().sum()} (mostly fines)")

## 6. Export Cleaned Dataset

In [ ]:
# Drop text columns and export
drop_cols = ['sentence_raw', 'extraction_notes']
export_df = df_sev.drop(columns=[c for c in drop_cols if c in df_sev.columns])

# Ensure figures directory exists
Path('../figures').mkdir(exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

export_path = Path('../data/processed/pilot_clean.parquet')
export_df.to_parquet(export_path, index=False)
print(f"Exported {len(export_df)} cases to {export_path}")
print(f"Columns: {list(export_df.columns)}")